# Vector counts after vectorisation

Takes one `pdf_dataset_stats.ipynb` run folder (`outputs/pdf_dataset_stats/<run_tag>/`) and, for every sampled `(pdf, page)` in its `sample_pages.json`, rebuilds the page the way the benchmark run saw it -- `Evaluation/conversion.py::convert_page_to_vector_text` (native text removed, drawn back as vector glyph paths, original drawings untouched), in memory, no files written -- and counts its vectors with the same `get_cdrawings()`/`get_drawings()` count the stats scan uses, so `num_vectors` (original) and `num_vectors_vectorised` are directly comparable.

Writes `vectorised_counts.csv` into the run folder and plots original vs vectorised counts, plus pipeline time / native→vector word recall against the vectorised count when the run has them.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / 'vectorised_counts.ipynb').exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import csv
import json

import matplotlib.pyplot as plt
import pymupdf as fitz

from rastervec.commons.logging_setup import configure_logging, get_logger
from rastervec.Evaluation import conversion, dump_io
from rastervec.Evaluation.Evaluate.timing import flatten_page_timing

configure_logging()
_LOG = get_logger('vectorised_counts')

## Parameters

In [ ]:
# A pdf_dataset_stats.ipynb run folder (the one holding sample_pages.json).
# None = the most recent run under outputs/pdf_dataset_stats/.
STATS_RUN_DIR = None

if STATS_RUN_DIR is None:
    runs = sorted(
        p for p in (PROJECT_ROOT / 'outputs' / 'pdf_dataset_stats').glob('*')
        if (p / 'sample_pages.json').is_file()
    )
    assert runs, 'no pdf_dataset_stats run with a sample_pages.json found'
    STATS_RUN_DIR = runs[-1]
STATS_RUN_DIR = Path(STATS_RUN_DIR)
assert (STATS_RUN_DIR / 'sample_pages.json').is_file(), f'no sample_pages.json in {STATS_RUN_DIR}'

samples = json.loads((STATS_RUN_DIR / 'sample_pages.json').read_text(encoding='utf-8'))
print('stats run dir:', STATS_RUN_DIR)
print(f'{len(samples)} sampled page(s)')

## 1. Vectorise each sampled page and count

Same counting as `pdf_dataset_stats.ipynb`'s scan (`get_cdrawings()` when available, else `get_drawings()`; one per drawing, i.e. one P1 `Vector` each). `num_native_text_vectorised` should be 0 -- it's a check that the conversion really removed the native text.

In [ ]:
_HAS_CDRAWINGS = hasattr(fitz.Page, 'get_cdrawings')


def _count_vectors(fitz_page) -> int:
    if _HAS_CDRAWINGS:
        return len(fitz_page.get_cdrawings(extended=False))
    return len(fitz_page.get_drawings(extended=False))


for i, r in enumerate(samples, 1):
    r['num_vectors_vectorised'] = None
    r['num_native_text_vectorised'] = None
    try:
        pdf_bytes = conversion.convert_page_to_vector_text(r['pdf_path'], r['page_index'])
        with fitz.open(stream=pdf_bytes, filetype='pdf') as doc:
            page = doc[0]
            r['num_vectors_vectorised'] = _count_vectors(page)
            r['num_native_text_vectorised'] = len(page.get_text('words'))
    except Exception as exc:  # keep going over the rest of the sample
        _LOG.warning('could not vectorise %s page %d: %s', r['pdf_path'], r['page_index'], exc)
    print(f'  [{i}/{len(samples)}] {Path(r["pdf_path"]).name} p{r["page_index"]}: '
          f'vectors {r["num_vectors"]} -> {r["num_vectors_vectorised"]} '
          f'(native text {r["num_native_text"]} -> {r["num_native_text_vectorised"]})')

counted = [r for r in samples if r['num_vectors_vectorised'] is not None]
print(f'{len(counted)}/{len(samples)} page(s) counted')

## 2. Pipeline time and accuracy (if the run has them)

Pipeline time comes from the stats run's own report (`pipeline_reports/*__generated_report_config/<stem>/dump.json`, via `flatten_page_timing`, the same as `pdf_dataset_stats.ipynb` step 5). Word recall is read from the run's `timing_results.txt` when that run scored native→vector accuracy.

In [ ]:
report_dirs = sorted((STATS_RUN_DIR / 'pipeline_reports').glob('*__generated_report_config'))
report_run_dir = report_dirs[-1] if report_dirs else None
print('report run dir:', report_run_dir)

dumps: dict[Path, object] = {}
for r in samples:
    r['total_time'] = None
    if report_run_dir is None:
        continue
    dump_path = report_run_dir / Path(r['pdf_path']).stem / 'dump.json'
    if not dump_path.is_file():
        continue
    if dump_path not in dumps:
        dumps[dump_path] = dump_io.load_dump(dump_path)
    page_dump = next((p for p in dumps[dump_path].pages if p.page_meta.index == r['page_index']), None)
    if page_dump is None:
        continue
    timing = flatten_page_timing(
        page_dump.step_durations, page_dump.substep_durations, page_dump.debug_durations,
    )
    r['total_time'] = timing.get('total') if timing else None

# timing_results.txt blocks: 'pdf: <path>' / '  page_index: N' / ... / '    word_recall: x'
recall_by_page: dict[tuple[str, int], float] = {}
results_path = STATS_RUN_DIR / 'timing_results.txt'
if results_path.is_file():
    pdf, page = None, None
    for line in results_path.read_text(encoding='utf-8').splitlines():
        s = line.strip()
        if line.startswith('pdf: '):
            pdf, page = line[len('pdf: '):], None
        elif s.startswith('page_index: '):
            page = int(s.split(': ', 1)[1])
        elif s.startswith('word_recall: ') and pdf is not None and page is not None:
            recall_by_page[(pdf, page)] = float(s.split(': ', 1)[1])
for r in samples:
    r['word_recall'] = recall_by_page.get((r['pdf_path'], r['page_index']))

print(f'{sum(r["total_time"] is not None for r in samples)} page(s) with timing, '
      f'{sum(r["word_recall"] is not None for r in samples)} with word recall')

## 3. Write `vectorised_counts.csv` and plot

In [ ]:
FIELDS = ['pdf_path', 'page_index', 'criterion', 'percentile', 'num_native_text', 'num_vectors',
          'num_vectors_vectorised', 'num_native_text_vectorised', 'total_time', 'word_recall']
csv_path = STATS_RUN_DIR / 'vectorised_counts.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(samples)
print('wrote', csv_path)

In [ ]:
CRITERION_COLORS = {
    'width_x_height': '#2563eb',
    'native_text_plus_vectors': '#f97316',
    'total_pixels': '#16a34a',
}
plot_dir = STATS_RUN_DIR / 'vectorised_plots'
plot_dir.mkdir(parents=True, exist_ok=True)


def _scatter(rows, x_key, y_key, xlabel, ylabel, fname, diagonal=False, ylim=None):
    rows = [r for r in rows if r.get(x_key) is not None and r.get(y_key) is not None]
    if not rows:
        print(f'skipping {fname}: no data')
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter([r[x_key] for r in rows], [r[y_key] for r in rows],
               c=[CRITERION_COLORS.get(r.get('criterion'), '#6b7280') for r in rows],
               s=60, edgecolor='white')
    if diagonal:
        hi = max(max(r[x_key] for r in rows), max(r[y_key] for r in rows))
        ax.plot([0, hi], [0, hi], color='#9ca3af', linewidth=1, linestyle='--')
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs {xlabel}')
    fig.tight_layout()
    fig.savefig(plot_dir / fname, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)


_scatter(samples, 'num_vectors', 'num_vectors_vectorised',
         'vectors (original)', 'vectors (vectorised)', 'vectorised_vs_original.png', diagonal=True)
_scatter(samples, 'num_vectors_vectorised', 'total_time',
         'vectors (vectorised)', 'pipeline time (s)', 'time_vs_vectorised_vectors.png')
_scatter(samples, 'num_vectors_vectorised', 'word_recall',
         'vectors (vectorised)', 'native->vector word recall', 'word_recall_vs_vectorised_vectors.png',
         ylim=(-0.05, 1.05))
print('wrote plots to', plot_dir)